In [3]:
import pandas as pd
df = pd.read_json('Varad_data.jsonl', lines=True)
print(df.head)

<bound method NDFrame.head of                                               messages
0    [{'role': 'user', 'content': 'intro'}, {'role'...
1    [{'role': 'user', 'content': 'Who is Varad?'},...
2    [{'role': 'user', 'content': 'Tell me about Va...
3    [{'role': 'user', 'content': 'What is Varad li...
4    [{'role': 'user', 'content': 'What makes Varad...
..                                                 ...
106  [{'role': 'user', 'content': 'Are you consciou...
107  [{'role': 'user', 'content': 'How was this AI ...
108  [{'role': 'user', 'content': 'What is Varad's ...
109  [{'role': 'user', 'content': 'Does Varad know ...
110  [{'role': 'user', 'content': 'What does Varad ...

[111 rows x 1 columns]>


In [4]:
df['messages'][1][1]['content']

"Varad Bendale. 2nd year Information Technology at K J Somaiya Institute of Technology, Mumbai. 9.57 CGPA — yeah that's not a typo. He's the kind of person who learns Random Forest and immediately goes 'cool, let me rebuild this in C++ from scratch with zero libraries.' That tells you everything. Genuinely passionate about AI and ML, not the surface level stuff — the deep, messy, how-does-this-actually-work kind. Wants to make real impact. Currently making me exist, which is either genius or madness, jury's still out."

In [5]:
import torch
import tiktoken
enc = tiktoken.encoding_for_model("gpt-4o")

special_tokens = {
    "<|user|>": 200100,
    "<|assistant|>": 200101,
    "<|end|>": 200102
}
allowed = set(special_tokens.keys())

user_messege = []
asistant_reply = []

for i in range(len(df['messages'])) :
    user_messege.append(df['messages'][i][0]['content'] )
    asistant_reply.append(df['messages'][i][1]['content'])

question_tokens  = []
target_tokens = []

for i in range(len(df['messages'])) :
    user_encods = enc.encode( "<|user|>"  + user_messege[i] ,  allowed_special= allowed)
    asis_encods = enc.encode( "<|assistant|>" +  asistant_reply[i] + "<|end|>"  , allowed_special= allowed )
    tokens = user_encods + asis_encods
    question = torch.tensor(tokens[:-1])
    question_tokens.append(question)
    target    = torch.tensor(tokens[1:])
    target_tokens.append(target)





In [ ]:
print(question_tokens)

In [6]:
g = torch.Generator()
g.manual_seed(42)

In [7]:
max_num = max(set(tokens))
print(max_num)

173781


In [8]:
vocab_size = len(set(tokens))
print(vocab_size)

107


In [9]:
compl_token = []
for i in range(len(question_tokens)):
    compl_token += question_tokens[i].tolist()
    compl_token += target_tokens[i].tolist()

vocab = sorted(set(compl_token))
stoi = {s:i for i,s in enumerate(vocab)}
itos = {i:s for i,s in enumerate(vocab)}
vocab_size = len(vocab)
q_tokens = []
t_tokens = []
for i in range(len(question_tokens)):
    q_tensor = torch.tensor([stoi[tok.item()] for tok in question_tokens[i]])
    t_tensor = torch.tensor([stoi[tok.item()] for tok in target_tokens[i]])
    q_tokens.append(q_tensor)
    t_tokens.append(t_tensor)

In [11]:
q_tokens = torch.cat(q_tokens)
t_tokens = torch.cat(t_tokens)

In [12]:
block_size = 3
xs = []
ys = []

for i in range(len(q_tokens)):
    context = q_tokens[max(0, i-block_size+1) : i+1]
    context = torch.nn.functional.pad(context, (block_size - len(context), 0))
    xs.append(context)
    ys.append(t_tokens[i])

xs = torch.stack(xs)
ys = torch.stack(ys)

In [13]:
dim = 100

Encods  = torch.randn((vocab_size, dim), requires_grad=True)
W1 = torch.randn((3*dim, dim), requires_grad=True)
W2 = torch.randn((dim, vocab_size), requires_grad=True)
enc = Encods[xs]
update_enc = enc.view(enc.shape[0] , -1 )
hidden = torch.tanh(update_enc @ W1 )
logits = hidden @ W2

In [14]:
b_gain = torch.ones((1, 100), requires_grad=True)
b_bias = torch.zeros((1, 100), requires_grad=True)
parameters  = { Encods , W1 , W2  , b_gain , b_bias}

In [15]:
logits.shape


torch.Size([15555, 2381])

In [16]:
loss = torch.nn.functional.cross_entropy(logits, ys)
print(loss)

tensor(34.3246, grad_fn=<NllLossBackward0>)


In [22]:
from google.colab import files
uploaded = files.upload()

checkpoint = torch.load('Amika.pt')
Encods      = checkpoint['Encods']
W1     = checkpoint['W1']
W2     = checkpoint['W2']
b_gain = checkpoint['b_gain']
b_bias = checkpoint['b_bias']
stoi   = checkpoint['stoi']
itos   = checkpoint['itos']

vocab_size = len(stoi)
parameters  = { Encods , W1 , W2  , b_gain , b_bias}

Saving Amika.pt to Amika (1).pt


In [40]:
print(parameters)

{tensor([[ 0.0400, -1.5779, -0.2044,  ..., -1.5261,  0.5104, -1.1427],
        [-1.1528, -0.8490, -1.1087,  ..., -1.4118, -0.5943, -0.8518],
        [-0.0175, -1.7609, -1.4941,  ..., -0.2305,  0.4015,  0.6439],
        ...,
        [-0.2254, -0.2041, -0.5948,  ..., -0.7951,  0.2903, -0.5783],
        [-1.7113, -1.7835, -0.1766,  ...,  0.7993, -0.5939, -0.5102],
        [ 0.1764,  1.1471,  0.5640,  ...,  0.8735,  1.8964, -0.9705]],
       requires_grad=True), tensor([[-0.1317,  0.0751, -0.0185, -0.1464, -0.0427,  0.0720, -0.0531,  0.0472,
          0.0142, -0.0466, -0.0196,  0.0577,  0.0258, -0.0304,  0.0557, -0.0021,
         -0.0082,  0.1163,  0.0497, -0.0940, -0.0892,  0.0798, -0.0769,  0.1130,
         -0.0990, -0.0846,  0.1117, -0.1297,  0.0115, -0.0996, -0.1041,  0.0405,
         -0.1219,  0.0504, -0.0209, -0.0864, -0.0564,  0.0943, -0.1000,  0.0511,
         -0.0781, -0.0664, -0.1066,  0.1283, -0.2294,  0.1438, -0.0461, -0.1916,
         -0.0514, -0.0264, -0.1695,  0.0347,  0.019

In [47]:
batch_size = 512
prev_loss = 100000
for i in range(10000):
    ix = torch.randint(0, xs.shape[0], (batch_size,))
    enc  = Encods[xs[ix]]
    update_enc = enc.view(enc.shape[0], -1)
    flow_layer = update_enc @ W1
    flow_layer = b_gain * (flow_layer - flow_layer.mean(0)) / flow_layer.std(0) + b_bias
    hidden     = torch.tanh(flow_layer)
    logits     = hidden @ W2
    loss       = torch.nn.functional.cross_entropy(logits, ys[ix])

    for p in parameters:
        p.grad = None
    loss.backward()
    lr = 0.01
    for p in parameters:
        p.data += -lr * p.grad

    if prev_loss < loss:
        prev_loss = loss
        break
print(loss)

tensor(5.4717, grad_fn=<NllLossBackward0>)


In [46]:

torch.save({
    'Encods': Encods,
    'W1': W1,
    'W2': W2,
    'b_gain': b_gain,
    'b_bias': b_bias,
    'stoi': stoi,
    'itos': itos
}, 'Amika.pt')


from google.colab import files
files.download('Amika.pt')

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>